# Hands-on Exercise 1 — Large-Scale ETL + Broadcast Join Comparison
### AI Operations (AIOps) — Module 2, Lecture 1 | ~45–55 minutes

**Referenced in:** *Module2_Slides_1_ApacheSpark.pptx*, Hands-on Exercise 1

**Objective:** process a large, partitioned clickstream dataset entirely with Spark DataFrames,
inspect the lazy execution plan, compute aggregated features, write partitioned Parquet output,
and measure the concrete cost difference between a naive join and a broadcast join.

**Steps (from the slide deck):**
1. Start a local SparkSession and read the provided partitioned clickstream Parquet dataset.
2. Build a `filter → groupBy → agg` pipeline computing `click_count` and `avg_session_length` per `(user_id, course_id)`.
3. Call `.explain(True)` **before** any action and identify the shuffle-inducing stage in the plan.
4. Write the result as partitioned Parquet, then join it against the small `course_catalog` lookup
   table twice: once naively, once with a broadcast join.
5. Compare shuffle read/write bytes for both joins using the Spark UI (`http://localhost:4040`).

**Deliverable:** a completed notebook showing the ETL pipeline, an annotated `.explain()` output
identifying the shuffle stage, and a before/after shuffle-bytes comparison for the two joins.

> **Prerequisites:** `pip install pyspark==3.5.*`, and a Java 11 or 17 JRE on PATH (verify with
> `java -version`). PySpark needs this even though you write Python.

## Step 0 — Generate the lab dataset

In a real classroom setting this dataset is pre-generated (10GB+) by the instructor ahead of time.
This cell generates a smaller, **scaled-down** synthetic version so the exercise runs quickly on a
laptop — the code and concepts are identical at any scale. Increase `N_ROWS` to simulate the full
10GB+ dataset if you have the disk space and time.

In [ ]:
import os, shutil

DATA_DIR = "data/clickstream"
CATALOG_DIR = "data/course_catalog"
OUTPUT_DIR = "output/features"

# Scaled down for a fast classroom lab. Increase to ~200_000_000 for a realistic 10GB+ run.
N_ROWS = 50_000_000
N_USERS = 50_000
N_COURSES = 200

for d in [DATA_DIR, CATALOG_DIR, OUTPUT_DIR]:
    shutil.rmtree(d, ignore_errors=True)

print(f"Will generate {N_ROWS:,} synthetic clickstream rows across {N_USERS:,} users and {N_COURSES} courses.")

## Step 1 — Start a local Spark session

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("aiops-module2-lecture1-etl")
    .config("spark.sql.shuffle.partitions", "8")   # tuned for a laptop-scale local cluster
    .config("spark.driver.memory", "4g")    # if you have less memory, reduce this value
    .config("spark.executor.memory", "4g")  # if you have less memory, reduce this value
    .config("spark.sql.autoBroadcastJoinThreshold", -1)  # disable auto-broadcast entirely
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## Step 1b — Materialize the synthetic datasets as partitioned Parquet

In [ ]:
from pyspark.sql import functions as F

# Raw clickstream events
raw = (
    spark.range(0, N_ROWS)
    .withColumn("user_id", (F.rand(seed=1) * N_USERS).cast("int"))
    .withColumn("course_id", (F.rand(seed=2) * N_COURSES).cast("int"))
    .withColumn(
        "event_type",
        F.when(F.rand(seed=3) < 0.7, F.lit("click")).otherwise(F.lit("view")),
    )
    .withColumn("session_length_sec", (F.rand(seed=4) * 600).cast("double"))
)
raw.write.mode("overwrite").parquet(DATA_DIR)

# Small lookup table -- deliberately tiny, a good broadcast-join candidate
catalog = spark.range(0, N_COURSES).withColumnRenamed("id", "course_id") \
    .withColumn("course_name", F.concat(F.lit("Course-"), F.col("course_id")))
catalog.write.mode("overwrite").parquet(CATALOG_DIR)

print("Synthetic datasets written.")

## Step 2 — Read the data and build the transformation pipeline

Remember: everything below is **lazy**. Nothing actually executes until an action is called.

In [ ]:
df = spark.read.parquet(DATA_DIR)
df.printSchema()

# print the number of rows
print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))
print("Number of distinct users:", df.select("user_id").distinct().count())
print("Number of distinct courses:", df.select("course_id").distinct().count())

print("Number of partitions:", df.rdd.getNumPartitions())

features = (
    df.filter(F.col("event_type") == "click")
      .groupBy("user_id", "course_id")
      .agg(
          F.count("*").alias("click_count"),
          F.avg("session_length_sec").alias("avg_session_length"),
      )
)
print("Pipeline built (nothing has executed yet).")

## Step 3 — Inspect the plan BEFORE any action runs

Look for an `Exchange` node in the physical plan — that is the shuffle caused by `groupBy`.

In [ ]:
features.explain(True)

> **Your annotation:** in the markdown cell below, note which line of the physical plan
> corresponds to the shuffle, and explain in one or two sentences WHY `groupBy` requires one
> (rows sharing a `(user_id, course_id)` key must land on the same executor to be aggregated together).

_Your answer here:_

- Shuffle-inducing stage: 
- Why it's needed: 

## Step 4 — Trigger execution and write partitioned Parquet output

In [ ]:
features.write.mode("overwrite").partitionBy("course_id").parquet(OUTPUT_DIR)
row_count = features.count()   # an ACTION
print(f"Rows written: {row_count:,}")

## Step 5 — Naive join vs. broadcast join

Open the Spark UI at **http://localhost:4040** (while this notebook's Spark session is alive) and
keep the **SQL / DataFrame** tab open while you run both cells below, so you can inspect the
shuffle read/write bytes for each query.

In [ ]:
catalog_df = spark.read.parquet(CATALOG_DIR)
catalog_df.show()

In [ ]:
#features.show()

In [ ]:
# --- Naive join: Spark may shuffle BOTH sides ---
joined_naive = features.join(catalog_df, "course_id")
joined_naive.explain("formatted")
naive_count = joined_naive.count()   # triggers execution -- check the Spark UI now
print("Naive join row count:", naive_count)

In [ ]:
from pyspark.sql.functions import broadcast

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10MB, catalog is tiny

joined_fast = features.join(broadcast(catalog_df), "course_id")

fast_count = joined_fast.count()   # trigger execution first
print("Broadcast join row count:", fast_count)

# Now inspect the FINAL adaptive plan (post-execution), which reflects what actually ran
joined_fast.explain("formatted")

## Step 6 — Record your comparison

In the Spark UI's SQL tab, open the query details for both jobs and note the **shuffle read** and
**shuffle write** bytes for each (not just wall-clock time — on a small local cluster, timing can be
noisy, but shuffle byte counts are a hard, provable difference).

_Your comparison here:_

| Join type | Shuffle read (bytes) | Shuffle write (bytes) |
|---|---|---|
| Naive join | | |
| Broadcast join | | |

**Observation:** 

## ✅ Deliverable Checklist
- [ ] A working `filter → groupBy → agg` pipeline producing `click_count` and `avg_session_length`
- [ ] `.explain(True)` output captured with the shuffle stage identified and explained
- [ ] Partitioned Parquet output written successfully
- [ ] A naive join and a broadcast join both executed, with a recorded shuffle-bytes comparison from the Spark UI

*Next: proceed to `Lecture2_Ray_Distributed_Exercise.ipynb`.*

In [ ]:
# Clean up the Spark session when you're done
spark.stop()